In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Update the path if your CSV is stored in a different folder.
df = pd.read_csv("Sample - Superstore.csv", encoding="cp1252")
print("Dataset shape:", df.shape)
display(df.head())
df.info()

customers = (
    df[["Customer ID", "Customer Name", "Segment"]]
    .drop_duplicates(subset="Customer ID")
)
orders = df[["Order ID", "Customer ID", "Region", "Sales"]].copy()
print("Customers:")
display(customers.head())
print("Orders:")
display(orders.head())

inner_merged = pd.merge(customers, orders, on="Customer ID", how="inner")
display(inner_merged.head())
print("INNER JOIN shape:", inner_merged.shape)

left_merged = pd.merge(customers, orders, on="Customer ID", how="left")
display(left_merged.head())
print("LEFT JOIN shape:", left_merged.shape)

right_merged = pd.merge(customers, orders, on="Customer ID", how="right")
display(right_merged.head())
print("RIGHT JOIN shape:", right_merged.shape)

outer_merged = pd.merge(customers, orders, on="Customer ID", how="outer", indicator=True)
display(outer_merged.head())
print("OUTER JOIN shape:", outer_merged.shape)
print(outer_merged["_merge"].value_counts())

sales_pivot = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)
display(sales_pivot)

# Use the pivot table without its totals row/column for a clean reshape.
wide_sales = sales_pivot.drop(index="Total", columns="Total").reset_index()
long_sales = pd.melt(
    wide_sales,
    id_vars=["Region"],
    var_name="Product",
    value_name="Sales"
)
display(wide_sales.head())
display(long_sales.head(10))
print("Wide shape:", wide_sales.shape)
print("Long shape:", long_sales.shape)

regional_category_sales = (
    df
    .loc[:, ["Region", "Category", "Sales", "Profit"]]
    .dropna(subset=["Region", "Category", "Sales"])
    .query("Sales > 0")
    .groupby(["Region", "Category"], as_index=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Sales=("Sales", "mean"),
        Transaction_Count=("Sales", "count")
    )
    .sort_values(by="Total_Sales", ascending=False)
    .reset_index(drop=True)
)
display(regional_category_sales.head(10))



Dataset shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
2,DV-13045,Darrin Van Huff,Corporate
3,SO-20335,Sean O'Donnell,Consumer
5,BH-11710,Brosina Hoffman,Consumer
12,AA-10480,Andrew Allen,Consumer


Orders:


,Order ID,Customer ID,Region,Sales
0,CA-2016-152156,CG-12520,South,261.9600
1,CA-2016-152156,CG-12520,South,731.9400
2,CA-2016-138688,DV-13045,West,14.6200
3,US-2015-108966,SO-20335,South,957.5775
4,US-2015-108966,SO-20335,South,22.3680


... (9 outputs left out)

Category,Region,Furniture,Office Supplies,Technology
0,Central,163797.1638,167026.415,170416.312
1,East,208291.2040,205516.055,264973.981
2,South,117298.6840,125651.313,148771.908
3,West,252612.7435,220853.249,251991.832


,Region,Product,Sales
0,Central,Furniture,163797.1638
1,East,Furniture,208291.2040
2,South,Furniture,117298.6840
3,West,Furniture,252612.7435
4,Central,Office Supplies,167026.4150
5,East,Office Supplies,205516.0550
6,South,Office Supplies,125651.3130
7,West,Office Supplies,220853.2490
8,Central,Technology,170416.3120
9,East,Technology,264973.9810


Wide shape: (4, 4)
Long shape: (12, 3)


,Region,Category,Total_Sales,Total_Profit,Average_Sales,Transaction_Count
0,East,Technology,264973.9810,47462.0351,495.278469,535
1,West,Furniture,252612.7435,11504.9503,357.302325,707
2,West,Technology,251991.8320,44303.6496,420.687533,599
3,West,Office Supplies,220853.2490,52609.8490,116.422377,1897
4,East,Furniture,208291.2040,3046.1658,346.574383,601
5,East,Office Supplies,205516.0550,41014.5791,120.044425,1712
6,Central,Technology,170416.3120,33697.4320,405.753124,420
7,Central,Office Supplies,167026.4150,8879.9799,117.458801,1422
8,Central,Furniture,163797.1638,-2871.0494,340.534644,481
9,South,Technology,148771.9080,19991.8314,507.753952,293
